# Landmarks4 Model Testing (No CY Offset)

This notebook loads `best_model.pt` from `checkpoints_landmarks4` and runs live landmark inference without any crop center Y offset.

In [1]:
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image

In [2]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, **kwargs):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        return torch.relu(self.bn(self.conv(x)))


class InceptionBlock(nn.Module):
    def __init__(
        self, in_channels, out_1x1, red_3x3, out_3x3, red_5x5, out_5x5, out_pool
    ):
        super().__init__()
        self.branch1 = ConvBlock(in_channels, out_1x1, kernel_size=1)
        self.branch2 = nn.Sequential(
            ConvBlock(in_channels, red_3x3, kernel_size=1, padding=0),
            ConvBlock(red_3x3, out_3x3, kernel_size=3, padding=1),
        )
        self.branch3 = nn.Sequential(
            ConvBlock(in_channels, red_5x5, kernel_size=1),
            ConvBlock(red_5x5, out_5x5, kernel_size=5, padding=2),
        )
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, padding=1, stride=1),
            ConvBlock(in_channels, out_pool, kernel_size=1),
        )

    def forward(self, x):
        branches = (self.branch1, self.branch2, self.branch3, self.branch4)
        return torch.cat([branch(x) for branch in branches], 1)


class InceptionModel(nn.Module):
    def __init__(self, num_classes=10, dropout=0.0):
        super().__init__()
        self.conv1 = ConvBlock(3, 32, kernel_size=3, stride=2, padding=1)
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = ConvBlock(32, 64, kernel_size=3, stride=1, padding=1)
        self.incept3a = InceptionBlock(64, 32, 32, 64, 16, 32, 32)
        self.incept3b = InceptionBlock(160, 64, 64, 96, 32, 64, 64)
        self.incept4a = InceptionBlock(288, 96, 64, 128, 32, 64, 64)
        self.incept4b = InceptionBlock(352, 96, 64, 128, 32, 64, 64)
        self.incept5a = InceptionBlock(352, 128, 96, 160, 32, 96, 96)
        self.dropout = nn.Dropout(p=dropout)
        self.fc = nn.Linear(480, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = self.incept3a(x)
        x = self.incept3b(x)
        x = self.maxpool(x)
        x = self.incept4a(x)
        x = self.incept4b(x)
        x = self.maxpool(x)
        x = self.incept5a(x)
        x = nn.functional.adaptive_avg_pool2d(x, (1, 1))
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        x = torch.sigmoid(x)
        return x

In [3]:
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "checkpoints_landmarks4":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "dl_lab345.ipynb" / "checkpoints_landmarks4"

CKPT_PATH = NOTEBOOK_DIR / "best_model.pt"
IMG_SIZE = 128
PIPELINE_SCALE = 1.35
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = InceptionModel(num_classes=10, dropout=0.0).to(DEVICE)
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

transform = transforms.Compose([transforms.ToTensor()])
print(f"Loaded checkpoint: {CKPT_PATH}")
print(f"Device: {DEVICE}")

Loaded checkpoint: /home/toru2/Amara/Deep_learning/dl_lab345.ipynb/checkpoints_landmarks4/best_model.pt
Device: cuda


In [4]:
def crop_face(frame_bgr, scale=1.35, cy_offset_ratio=0.0):
    h, w = frame_bgr.shape[:2]
    side = min(h, w)
    side = int(side / scale)

    x = (w - side) // 2
    y = (h - side) // 2

    fh = float(side)
    cy = y + fh / 2.0 + (cy_offset_ratio * fh)
    y = int(round(cy - fh / 2.0))

    x = max(0, min(x, w - side))
    y = max(0, min(y, h - side))

    face = frame_bgr[y:y+side, x:x+side]
    return face, (x, y, x + side, y + side)


def predict_landmarks(face_resized_rgb):
    inp = transform(Image.fromarray(face_resized_rgb)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = model(inp).squeeze(0).cpu().numpy().reshape(5, 2)
    return np.clip(pred, 0.0, 1.0)


def draw_landmarks(frame_bgr, bbox_xyxy, pred_norm, color=(0, 255, 0)):
    x1, y1, x2, y2 = bbox_xyxy
    fw = x2 - x1
    fh = y2 - y1
    for px, py in pred_norm:
        cx = int(x1 + px * fw)
        cy = int(y1 + py * fh)
        cv2.circle(frame_bgr, (cx, cy), 2, color, -1, lineType=cv2.LINE_AA)
    cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (255, 180, 0), 1, lineType=cv2.LINE_AA)

In [8]:
# Video testing loop (press 'q' to quit)
VIDEO_PATH = "/home/toru2/Amara/Deep_learning/dl_lab345.ipynb/video/eyeGlass.mov"

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

while True:
    ok, frame = cap.read()
    if not ok:
        break

    face_crop, bbox = crop_face(frame, scale=PIPELINE_SCALE, cy_offset_ratio=0.0)
    face_resized = cv2.resize(face_crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    face_resized_rgb = cv2.cvtColor(face_resized, cv2.COLOR_BGR2RGB)

    pred = predict_landmarks(face_resized_rgb)
    draw_landmarks(frame, bbox, pred)

    cv2.putText(frame, "No CY offset", (12, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(frame, "test1.mov", (12, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 220, 120), 2)
    cv2.imshow("Landmarks4 Test (Video)", frame)

    if (cv2.waitKey(1) & 0xFF) == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
print(f"Done. Used video={VIDEO_PATH}, scale={PIPELINE_SCALE}, cy_off=0.0")

Done. Used video=/home/toru2/Amara/Deep_learning/dl_lab345.ipynb/video/eyeGlass.mov, scale=1.35, cy_off=0.0
